# Writing and reading YAML

`mammos_entity` can write and read an `EntityCollection` in YAML format with a custom structure. Details of the file format are explained in the [`mammos_entity.EntityCollection.to_csv` api reference](https://mammos-project.github.io/mammos/api/mammos_entity.html#mammos_entity.EntityCollection.to_yaml). YAML is the most flexible text-based format supported by `mammos-entity`.

In [1]:
from pathlib import Path

import mammos_entity as me
from mammos_entity import units as u

## Writing

We create some artificial data that we can write to file. In our collection we have entity-likes with different length and shape.

In [2]:
collection = me.EntityCollection(
    "Example description.\nThe description can have multiple lines.\n\nLines can also be empty.",
    Ms=me.Ms([700, 650, 600], "kA/m", description="Evaluated using UppASD with 70000 Monte Carlo steps."),
    T=me.T([100, 200, 300], "K"),
    Tc=me.Tc(600, "K"),
    theta_angle=[[0, 0.5], [0.7, 1]] * u.rad,
    demag_factor=me.Entity("DemagnetizingFactor", [1 / 3, 1 / 3, 1 / 3]),
    comments=["Some comment", "Some other comment", "A third comment"],
)
collection

EntityCollection(
    description='Example description.\nThe description can have multiple lines.\n\nLines can also be empty.',
    Ms=Entity(ontology_label='SpontaneousMagnetization', value=array([700., 650., 600.]), unit='kA / m', description='Evaluated using UppASD with 70000 Monte Carlo steps.'),
    T=Entity(ontology_label='ThermodynamicTemperature', value=array([100., 200., 300.]), unit='K'),
    Tc=Entity(ontology_label='CurieTemperature', value=600.0, unit='K'),
    theta_angle=<Quantity [[0. , 0.5],
               [0.7, 1. ]] rad>,
    demag_factor=Entity(ontology_label='DemagnetizingFactor', value=array([0.33333333, 0.33333333, 0.33333333])),
    comments=['Some comment', 'Some other comment', 'A third comment'],
)

We can write data to a yaml file as shown in the following cell: 

In [3]:
collection.to_yaml("example.yaml")

This has produced the following file:

In [4]:
print(Path("example.yaml").read_text())

metadata:
  version: v2
description: |-
  Example description.
  The description can have multiple lines.

  Lines can also be empty.
data:
  Ms:
    ontology_label: SpontaneousMagnetization
    description: Evaluated using UppASD with 70000 Monte Carlo steps.
    ontology_iri: https://w3id.org/emmo/domain/magnetic_material#EMMO_032731f8-874d-5efb-9c9d-6dafaa17ef25
    unit: kA / m
    value: [700.0, 650.0, 600.0]
  T:
    ontology_label: ThermodynamicTemperature
    description: ''
    ontology_iri: https://w3id.org/emmo#EMMO_affe07e4_e9bc_4852_86c6_69e26182a17f
    unit: K
    value: [100.0, 200.0, 300.0]
  Tc:
    ontology_label: CurieTemperature
    description: ''
    ontology_iri: https://w3id.org/emmo#EMMO_6b5af5a8_a2d8_4353_a1d6_54c9f778343d
    unit: K
    value: 600.0
  theta_angle:
    unit: rad
    value: [[0.0, 0.5], [0.7, 1.0]]
  demag_factor:
    ontology_label: DemagnetizingFactor
    description: ''
    ontology_iri: https://w3id.org/emmo/domain/magnetic_material#EMMO_

### Reading
We can read it back in to recreate the original `EntityCollection`:

In [5]:
content = me.from_yaml("example.yaml")
content

EntityCollection(
    description='Example description.\nThe description can have multiple lines.\n\nLines can also be empty.',
    Ms=Entity(ontology_label='SpontaneousMagnetization', value=array([700., 650., 600.]), unit='kA / m', description='Evaluated using UppASD with 70000 Monte Carlo steps.'),
    T=Entity(ontology_label='ThermodynamicTemperature', value=array([100., 200., 300.]), unit='K'),
    Tc=Entity(ontology_label='CurieTemperature', value=600.0, unit='K'),
    theta_angle=<Quantity [[0. , 0.5],
               [0.7, 1. ]] rad>,
    demag_factor=Entity(ontology_label='DemagnetizingFactor', value=array([0.33333333, 0.33333333, 0.33333333])),
    comments=['Some comment', 'Some other comment', 'A third comment'],
)

As shown in the [`EntityCollection` notebook](entity-collection.ipynb) we can access the individual elements of the collection:

In [6]:
content.Ms

Entity(ontology_label='SpontaneousMagnetization', value=array([700., 650., 600.]), unit='kA / m', description='Evaluated using UppASD with 70000 Monte Carlo steps.')

In [7]:
content.theta_angle

<Quantity [[0. , 0.5],
           [0.7, 1. ]] rad>

## Nested collections

An element of an `EntityCollection` can itself also be an `EntityCollection`.

To demonstrate that, let us create a second collection:

In [8]:
inner_collection = me.EntityCollection(
    A=me.A(1e-11, "J/m"), Ms=me.Ms(800, "kA/m"), description="Zero-temperature properties"
)
inner_collection

EntityCollection(
    description='Zero-temperature properties',
    A=Entity(ontology_label='ExchangeStiffnessConstant', value=1e-11, unit='J / m'),
    Ms=Entity(ontology_label='SpontaneousMagnetization', value=800.0, unit='kA / m'),
)

We can add it to the collection created at the top of this notebook:

In [9]:
collection.intrinsic_properties = inner_collection
collection

EntityCollection(
    description='Example description.\nThe description can have multiple lines.\n\nLines can also be empty.',
    Ms=Entity(ontology_label='SpontaneousMagnetization', value=array([700., 650., 600.]), unit='kA / m', description='Evaluated using UppASD with 70000 Monte Carlo steps.'),
    T=Entity(ontology_label='ThermodynamicTemperature', value=array([100., 200., 300.]), unit='K'),
    Tc=Entity(ontology_label='CurieTemperature', value=600.0, unit='K'),
    theta_angle=<Quantity [[0. , 0.5],
               [0.7, 1. ]] rad>,
    demag_factor=Entity(ontology_label='DemagnetizingFactor', value=array([0.33333333, 0.33333333, 0.33333333])),
    comments=['Some comment', 'Some other comment', 'A third comment'],
    intrinsic_properties=EntityCollection(
        description='Zero-temperature properties',
        A=Entity(ontology_label='ExchangeStiffnessConstant', value=1e-11, unit='J / m'),
        Ms=Entity(ontology_label='SpontaneousMagnetization', value=800.0, unit='kA /

an save the collection to disk:

In [10]:
collection.to_yaml("example-nested.yaml")

The file with nested collections looks as follows:

In [11]:
print(Path("example-nested.yaml").read_text())

metadata:
  version: v2
description: |-
  Example description.
  The description can have multiple lines.

  Lines can also be empty.
data:
  Ms:
    ontology_label: SpontaneousMagnetization
    description: Evaluated using UppASD with 70000 Monte Carlo steps.
    ontology_iri: https://w3id.org/emmo/domain/magnetic_material#EMMO_032731f8-874d-5efb-9c9d-6dafaa17ef25
    unit: kA / m
    value: [700.0, 650.0, 600.0]
  T:
    ontology_label: ThermodynamicTemperature
    description: ''
    ontology_iri: https://w3id.org/emmo#EMMO_affe07e4_e9bc_4852_86c6_69e26182a17f
    unit: K
    value: [100.0, 200.0, 300.0]
  Tc:
    ontology_label: CurieTemperature
    description: ''
    ontology_iri: https://w3id.org/emmo#EMMO_6b5af5a8_a2d8_4353_a1d6_54c9f778343d
    unit: K
    value: 600.0
  theta_angle:
    unit: rad
    value: [[0.0, 0.5], [0.7, 1.0]]
  demag_factor:
    ontology_label: DemagnetizingFactor
    description: ''
    ontology_iri: https://w3id.org/emmo/domain/magnetic_material#EMMO_